# Blood Donor Retention Prediction — Reproducible Analysis

Samarpan Blood Bank synthetic dataset. Research-guided RFMTC features, 180-day retention and 365-day churn targets.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.data.loader import load_donor_donation_tables
from src.data.cleaning import clean_donors, clean_donations, align_donations_to_donors
from src.features.engineering import build_observation_dataset, prepare_model_matrix
from src.segmentation.rfm import build_rfm_table, segment_summary

In [ ]:
donors_raw, donations_raw = load_donor_donation_tables(str(PROJECT_ROOT / 'data' / 'Samarpan_BloodBank_SyntheticDataset_V2.xlsx'))
donors = clean_donors(donors_raw)
donations = clean_donations(donations_raw)
donors, donations = align_donations_to_donors(donors, donations)
print(donors.shape, donations.shape)
donations.head()

In [ ]:
dataset = build_observation_dataset(donors, donations)
print('Observations:', len(dataset))
print('Retention rate (180d):', dataset['retained_180'].mean())
print('Churn rate (365d):', dataset['churn_365'].mean())
dataset[['Donor_ID', 'anchor_date', 'total_donations', 'days_since_last_donation', 'retained_180', 'churn_365']].head()

In [ ]:
rfm = build_rfm_table(donors, donations)
segment_summary(rfm)

In [ ]:
import json
metrics_dir = PROJECT_ROOT / 'outputs' / 'metrics'
for target in ['retained_180', 'churn_365']:
    with open(metrics_dir / f'{target}_leaderboard.json') as f:
        board = json.load(f)
    print(f'\n=== {target} leaderboard ===')
    display(pd.DataFrame(board))

In [ ]:
figures = sorted((PROJECT_ROOT / 'outputs' / 'figures').glob('*.png'))
for fig_path in figures[:6]:
    print(fig_path.name)
    display(Image(filename=str(fig_path)))

## Re-run Full Pipeline

```bash
python scripts/run_pipeline.py
```